<a href="https://colab.research.google.com/github/Miralles-Iborra/Biomec-Lab-Tools-ES/blob/main/PlataformaFuerza_Salto_HerramientaInteractiva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Análisis interactivo de saltos con plataformas de fuerza

Este notebook permite a una persona sin experiencia en programación:

1. Subir un **TXT individual exportado desde Kistler BioWare** o una **matriz CSV** con varios archivos.
2. Seleccionar y verificar el salto.
3. Calcular:
   - fuerza vertical total (N);
   - fuerza normalizada al peso corporal (BW);
   - aceleración vertical del centro de masas (m·s⁻²);
   - velocidad vertical del centro de masas (m·s⁻¹);
   - desplazamiento vertical del centro de masas (m).
4. Explorar una gráfica interactiva superpuesta, con selección individual de variables, zoom, desplazamiento y valores al pasar el cursor.
5. Descargar los datos procesados.

> **Convención:** los valores positivos de aceleración, velocidad y desplazamiento indican movimiento hacia arriba.

> **Carga en Colab:** ejecute manualmente la celda 2 cada vez que abra o reconecte el notebook. El selector de archivos pertenece a la sesión actual del navegador y no se conserva entre sesiones.

> **Importante para DJ:** si el salto comienza cayendo desde un cajón, debe indicarse la altura de caída. Sin esa información, la velocidad inicial de contacto no puede estimarse correctamente solo con la fuerza.


In [ ]:

#@title 1. Preparar el entorno y las funciones (ejecutar una sola vez)
!pip -q install ipywidgets plotly scipy

import os
import re
import io
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import ipywidgets as widgets
import plotly.graph_objects as go
import plotly.io as pio

from IPython.display import display, clear_output, HTML
from scipy.signal import butter, filtfilt
from scipy.integrate import cumulative_trapezoid
from plotly.subplots import make_subplots

try:
    from google.colab import output, files
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
    files = None
    pio.renderers.default = "notebook_connected"

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

G = 9.80665

STATE = {
    "source_kind": None,
    "source_path": None,
    "source_name": None,
    "txt_metadata": {},
    "catalog": None,
    "selected_id": None,
    "selected_metadata": {},
    "selected_trial": None,
    "processed": None,
    "events": None,
    "metrics": None,
    "drop_height_m": None,
}

def _normalise_name(text):
    return re.sub(r"[^a-z0-9]+", "", str(text).strip().lower())

def _safe_float(value, default=np.nan):
    try:
        return float(str(value).replace(",", "."))
    except Exception:
        return default

def _guess_separator(path):
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        first = f.readline()
    counts = {";": first.count(";"), "\t": first.count("\t"), ",": first.count(",")}
    return max(counts, key=counts.get)

def _extract_uploaded_file(file_upload_widget):
    value = file_upload_widget.value
    if not value:
        raise ValueError("No se ha seleccionado ningún archivo.")

    if isinstance(value, dict):  # ipywidgets 7
        name, item = next(iter(value.items()))
        content = item.get("content", item)
        name = item.get("name", name)
    else:  # ipywidgets 8
        item = value[0]
        name = item["name"]
        content = item["content"]

    if hasattr(content, "tobytes"):
        content = content.tobytes()

    destination = Path("/content" if Path("/content").exists() else ".") / Path(name).name
    destination.write_bytes(content)
    return str(destination), Path(name).name

def parse_kistler_txt(path):
    """Lee TXT de BioWare con 1 o más plataformas de tres componentes."""
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        lines = f.read().splitlines()

    header = {}
    data_header_index = None

    for i, line in enumerate(lines):
        if line.lower().startswith("abs time"):
            data_header_index = i
            break
        if "\t" in line:
            parts = line.split("\t")
            key = parts[0].strip().rstrip(":")
            header[key] = [p.strip() for p in parts[1:]]

    if data_header_index is None:
        raise ValueError("No se encontró la cabecera 'abs time (s)' en el TXT.")

    data_start = data_header_index + 2  # omite nombres y unidades
    numeric_rows = []
    expected_columns = None

    for line in lines[data_start:]:
        if not line.strip():
            continue
        parts = [p.strip() for p in line.split("\t")]
        try:
            values = [float(p.replace(",", ".")) for p in parts if p != ""]
        except ValueError:
            continue
        if expected_columns is None:
            expected_columns = len(values)
        if len(values) == expected_columns:
            numeric_rows.append(values)

    if not numeric_rows or expected_columns is None or expected_columns < 4:
        raise ValueError("No se pudieron leer datos numéricos de fuerza en el TXT.")

    n_force_columns = expected_columns - 1
    if n_force_columns % 3 != 0:
        raise ValueError(
            f"Se encontraron {n_force_columns} columnas de fuerza; se esperaban grupos Fx/Fy/Fz."
        )

    n_platforms = n_force_columns // 3
    columns = ["tiempo_s"]
    for p in range(1, n_platforms + 1):
        columns += [f"Fx_P{p}", f"Fy_P{p}", f"Fz_P{p}"]

    trial = pd.DataFrame(numeric_rows, columns=columns)
    trial["Fz_total"] = trial[[f"Fz_P{p}" for p in range(1, n_platforms + 1)]].sum(axis=1)

    rate_values = header.get("Rate (Hz)", [])
    normalised_force_values = header.get("Normalized force (N)", [])

    fs = _safe_float(rate_values[0]) if rate_values else np.nan
    if not np.isfinite(fs):
        dt = np.nanmedian(np.diff(trial["tiempo_s"]))
        fs = 1.0 / dt

    metadata = {
        "archivo": Path(path).name,
        "frecuencia_hz": float(fs),
        "n_plataformas": int(n_platforms),
        "peso_corporal_metadata_N": (
            _safe_float(normalised_force_values[0]) if normalised_force_values else np.nan
        ),
        "duracion_s": float(trial["tiempo_s"].iloc[-1] - trial["tiempo_s"].iloc[0]),
        "n_muestras": int(len(trial)),
        "tipo_salto": "",
    }
    return trial, metadata, header

def build_csv_catalog(path, chunksize=100_000):
    """Crea un catálogo de saltos sin cargar toda la matriz en memoria."""
    sep = _guess_separator(path)
    columns = pd.read_csv(path, sep=sep, nrows=0, encoding="utf-8-sig").columns.tolist()
    norm = {_normalise_name(c): c for c in columns}

    id_col = norm.get("idarchivo") or norm.get("rutarelativa") or norm.get("archivo")
    if id_col is None:
        raise ValueError(
            "La matriz necesita una columna identificadora como 'id_archivo', "
            "'ruta_relativa' o 'archivo'."
        )

    preferred = [
        id_col, "archivo", "anio", "grupo", "carpeta_grupo", "codigo_sujeto",
        "tipo_salto", "intento", "fecha_medicion", "frecuencia_hz",
        "n_plataformas", "ruta_relativa"
    ]
    usecols = []
    for wanted in preferred:
        match = next((c for c in columns if _normalise_name(c) == _normalise_name(wanted)), None)
        if match and match not in usecols:
            usecols.append(match)

    pieces = []
    for chunk in pd.read_csv(
        path, sep=sep, usecols=usecols, chunksize=chunksize,
        encoding="utf-8-sig", low_memory=False
    ):
        chunk[id_col] = chunk[id_col].astype(str)
        pieces.append(chunk.drop_duplicates(subset=[id_col]))

    catalog = pd.concat(pieces, ignore_index=True).drop_duplicates(subset=[id_col])
    catalog = catalog.rename(columns={id_col: "id_archivo_selector"})

    # Añade columnas ausentes para simplificar la interfaz.
    for c in ["archivo", "anio", "grupo", "codigo_sujeto", "tipo_salto",
              "intento", "frecuencia_hz", "n_plataformas", "fecha_medicion"]:
        if c not in catalog.columns:
            catalog[c] = ""

    catalog["id_archivo_selector"] = catalog["id_archivo_selector"].astype(str)
    catalog = catalog.sort_values(
        ["anio", "grupo", "codigo_sujeto", "tipo_salto", "intento", "archivo"],
        kind="stable"
    ).reset_index(drop=True)

    return catalog, sep

def load_csv_trial(path, trial_id, chunksize=100_000):
    sep = _guess_separator(path)
    columns = pd.read_csv(path, sep=sep, nrows=0, encoding="utf-8-sig").columns.tolist()
    norm = {_normalise_name(c): c for c in columns}
    id_col = norm.get("idarchivo") or norm.get("rutarelativa") or norm.get("archivo")

    frames = []
    for chunk in pd.read_csv(
        path, sep=sep, chunksize=chunksize,
        encoding="utf-8-sig", low_memory=False
    ):
        mask = chunk[id_col].astype(str) == str(trial_id)
        if mask.any():
            frames.append(chunk.loc[mask].copy())

    if not frames:
        raise ValueError(f"No se encontró el salto '{trial_id}' en la matriz.")

    trial = pd.concat(frames, ignore_index=True)
    return standardise_trial(trial)

def standardise_trial(df):
    """Normaliza nombres de tiempo y fuerza sin modificar las columnas originales."""
    data = df.copy()
    norm = {_normalise_name(c): c for c in data.columns}

    time_col = (
        norm.get("tiempos") or norm.get("abstimes") or norm.get("time")
        or norm.get("times") or norm.get("tiempo")
    )
    if time_col is None:
        sample_col = norm.get("muestra") or norm.get("sample")
        fs_col = norm.get("frecuenciahz") or norm.get("ratehz")
        if sample_col is None or fs_col is None:
            raise ValueError("No se encontró una columna de tiempo ni muestra + frecuencia.")
        data["tiempo_s"] = (
            pd.to_numeric(data[sample_col], errors="coerce")
            / pd.to_numeric(data[fs_col], errors="coerce")
        )
    else:
        data["tiempo_s"] = pd.to_numeric(data[time_col], errors="coerce")

    fz_total_col = norm.get("fztotal")
    if fz_total_col is not None:
        data["Fz_total"] = pd.to_numeric(data[fz_total_col], errors="coerce")
    else:
        fz_cols = [
            c for c in data.columns
            if re.fullmatch(r"fz[_\s-]*p?\d+", str(c).strip(), flags=re.I)
        ]
        if not fz_cols:
            # También admite un único Fz.
            single_fz = next(
                (c for c in data.columns if _normalise_name(c) in {"fz", "forcez", "verticalforce"}),
                None
            )
            if single_fz is None:
                raise ValueError("No se encontró 'Fz_total' ni columnas verticales Fz.")
            fz_cols = [single_fz]
        numeric_fz = data[fz_cols].apply(pd.to_numeric, errors="coerce")
        data["Fz_total"] = numeric_fz.sum(axis=1, min_count=1)

    data = data.dropna(subset=["tiempo_s", "Fz_total"]).sort_values("tiempo_s")
    data = data.drop_duplicates(subset=["tiempo_s"]).reset_index(drop=True)

    if len(data) < 20:
        raise ValueError("El salto contiene muy pocas muestras válidas.")

    return data

def get_trial_metadata(trial, fallback=None):
    fallback = fallback or {}
    meta = dict(fallback)

    def first_existing(names, default=""):
        for name in names:
            if name in trial.columns and trial[name].notna().any():
                return trial[name].dropna().iloc[0]
        return default

    dt = np.nanmedian(np.diff(trial["tiempo_s"].to_numpy(dtype=float)))
    fs_from_time = 1.0 / dt if np.isfinite(dt) and dt > 0 else np.nan

    fs = first_existing(["frecuencia_hz", "Rate (Hz)", "rate_hz"], fs_from_time)
    meta.update({
        "archivo": first_existing(["archivo"], meta.get("archivo", "")),
        "id_archivo": first_existing(
            ["id_archivo", "ruta_relativa"], meta.get("id_archivo", "")
        ),
        "anio": first_existing(["anio"], meta.get("anio", "")),
        "grupo": first_existing(["grupo"], meta.get("grupo", "")),
        "codigo_sujeto": first_existing(
            ["codigo_sujeto"], meta.get("codigo_sujeto", "")
        ),
        "tipo_salto": first_existing(["tipo_salto"], meta.get("tipo_salto", "")),
        "intento": first_existing(["intento"], meta.get("intento", "")),
        "frecuencia_hz": float(_safe_float(fs, fs_from_time)),
        "n_muestras": int(len(trial)),
        "duracion_s": float(
            trial["tiempo_s"].iloc[-1] - trial["tiempo_s"].iloc[0]
        ),
    })
    return meta

def lowpass(signal, fs, cutoff=20.0, order=4):
    x = pd.Series(np.asarray(signal, dtype=float)).interpolate(
        limit_direction="both"
    ).to_numpy()

    if cutoff <= 0 or fs <= 2 * cutoff or len(x) < 30:
        return x

    b, a = butter(order, cutoff / (fs / 2), btype="low")
    padlen = min(3 * max(len(a), len(b)), len(x) - 1)
    return filtfilt(b, a, x, padlen=padlen)

def force_sign_correct(force):
    f = np.asarray(force, dtype=float)
    p5, p95 = np.nanpercentile(f, [5, 95])
    if abs(p5) > abs(p95) and p95 < 100:
        return -f, True
    return f, False

def sustained_index(mask, n_samples, start=0):
    mask = np.asarray(mask, dtype=bool)
    n_samples = max(int(n_samples), 1)
    start = max(int(start), 0)

    if len(mask) < n_samples or start >= len(mask):
        return None

    convolution = np.convolve(
        mask.astype(np.int8),
        np.ones(n_samples, dtype=np.int16),
        mode="valid"
    )
    indices = np.flatnonzero(convolution[start:] >= n_samples)
    return int(start + indices[0]) if indices.size else None

def estimate_bw_and_quiet(time, force, fs, prefer_early=True):
    """
    Estima BW buscando 0.5 s de apoyo estable.
    En CMJ/SJ prioriza el primer tramo estable; en DJ permite usar el tramo posterior.
    """
    t = np.asarray(time, dtype=float)
    f = np.asarray(force, dtype=float)
    n = max(int(round(0.5 * fs)), 50)
    n = min(n, len(f))
    step = max(int(round(0.05 * fs)), 1)

    candidates = []
    for i in range(0, len(f) - n + 1, step):
        segment = f[i:i+n]
        mean = np.nanmean(segment)
        sd = np.nanstd(segment)
        median = np.nanmedian(segment)

        if not np.isfinite(mean) or not 250 <= mean <= 2000:
            continue

        cv = sd / max(abs(mean), 1)
        x = t[i:i+n]
        if np.ptp(x) > 0:
            slope = np.polyfit(x, segment, 1)[0]
            relative_slope = abs(slope) / max(abs(mean), 1)
        else:
            relative_slope = np.inf

        if cv <= 0.035 and relative_slope <= 0.05:
            score = cv + 0.2 * relative_slope
            candidates.append((i, i+n, median, sd, score))

    if not candidates:
        for i in range(0, len(f) - n + 1, step):
            segment = f[i:i+n]
            median = np.nanmedian(segment)
            if 250 <= median <= 2000:
                mad = np.nanmedian(np.abs(segment - median))
                candidates.append((i, i+n, median, np.nanstd(segment), mad / median))

    if not candidates:
        raise ValueError(
            "No se pudo localizar automáticamente un periodo estable de apoyo. "
            "Introduzca manualmente la masa corporal o el peso corporal."
        )

    if prefer_early:
        early_limit = int(0.45 * len(f))
        early = [c for c in candidates if c[0] <= early_limit]
        pool = early if early else candidates
        # El primer periodo suficientemente estable es preferible en CMJ/SJ.
        best = sorted(pool, key=lambda c: (c[0], c[4]))[0]
    else:
        # En DJ, el sujeto puede no estar sobre la plataforma al inicio.
        best = sorted(candidates, key=lambda c: c[4])[0]

    i, j, bw, sd, _ = best
    return float(bw), (int(i), int(j)), float(sd)

def detect_quiet_start_events(time, force_filtered, bw, fs, quiet_idx):
    t = np.asarray(time, dtype=float)
    f = np.asarray(force_filtered, dtype=float)
    qs, qe = quiet_idx

    quiet_sd = float(np.nanstd(f[qs:qe]))
    movement_threshold = max(0.05 * bw, 5 * quiet_sd, 20.0)
    deviation = np.abs(f - bw) > movement_threshold

    onset = sustained_index(
        deviation,
        max(int(round(0.030 * fs)), 1),
        start=qe
    )
    if onset is None:
        raise ValueError("No se pudo detectar el inicio del movimiento.")

    contact_threshold = max(20.0, 0.03 * bw)
    takeoff = sustained_index(
        f < contact_threshold,
        max(int(round(0.015 * fs)), 1),
        start=min(onset + int(0.10 * fs), len(f) - 1)
    )

    landing = None
    if takeoff is not None:
        landing = sustained_index(
            f > contact_threshold,
            max(int(round(0.015 * fs)), 1),
            start=min(takeoff + int(0.05 * fs), len(f) - 1)
        )

    return {
        "onset": onset,
        "contact": None,
        "takeoff": takeoff,
        "landing": landing,
        "quiet_start": qs,
        "quiet_end": qe,
        "movement_threshold_N": movement_threshold,
        "contact_threshold_N": contact_threshold,
    }


def analyse_quiet_start(trial, fs, bw, cutoff, quiet_idx):
    """
    Calcula la cinemática vertical del COM en un CMJ/SJ.

    Antes del inicio del movimiento el sujeto se considera inmóvil, por lo que
    velocidad y desplazamiento se exportan como 0 en vez de NaN. La integración
    comienza exactamente en el inicio detectado para evitar acumular ruido del
    periodo de pesaje.
    """
    time = trial["tiempo_s"].to_numpy(dtype=float)
    force_raw, inverted = force_sign_correct(
        trial["Fz_total"].to_numpy(dtype=float)
    )
    force_filtered = lowpass(force_raw, fs, cutoff)
    events = detect_quiet_start_events(
        time, force_filtered, bw, fs, quiet_idx
    )

    mass = bw / G
    acceleration = (force_filtered - bw) / mass

    # Elimina el pequeño offset residual usando el periodo estable de pesaje.
    qs, qe = events["quiet_start"], events["quiet_end"]
    acceleration_offset = float(np.nanmedian(acceleration[qs:qe]))
    acceleration = acceleration - acceleration_offset

    onset = int(events["onset"])
    velocity = np.zeros(len(time), dtype=float)
    displacement = np.zeros(len(time), dtype=float)

    velocity[onset:] = cumulative_trapezoid(
        acceleration[onset:], time[onset:], initial=0.0
    )
    displacement[onset:] = cumulative_trapezoid(
        velocity[onset:], time[onset:], initial=0.0
    )

    if not np.isfinite(velocity).all() or not np.isfinite(displacement).all():
        raise ValueError(
            "La integración produjo valores no finitos. Compruebe tiempo, fuerza y BW."
        )

    return force_filtered, acceleration, velocity, displacement, events, inverted



def analyse_drop_jump(trial, fs, bw, cutoff, drop_height_m):
    """
    Calcula la cinemática de un drop jump mediante impulso-momento.

    Se generan dos referencias de posición:
    1) desplazamiento relativo al primer contacto, integrado desde la fuerza;
    2) posición reconstruida desde la liberación, incorporando una caída
       balística estimada a partir de la altura introducida.

    La altura debe interpretarse idealmente como descenso vertical efectivo del
    COM. La altura del cajón puede utilizarse como aproximación, pero no siempre
    coincide exactamente con el descenso real del COM.
    """
    if not np.isfinite(drop_height_m) or drop_height_m <= 0:
        raise ValueError(
            "Para un DJ debe introducirse una altura de caída mayor que 0 cm."
        )

    time = trial["tiempo_s"].to_numpy(dtype=float)
    force_raw, inverted = force_sign_correct(
        trial["Fz_total"].to_numpy(dtype=float)
    )
    force_filtered = lowpass(force_raw, fs, cutoff)

    mass = bw / G
    acceleration = (force_filtered - bw) / mass

    contact_threshold = max(20.0, 0.03 * bw)
    min_contact_samples = max(int(round(0.010 * fs)), 1)
    min_flight_samples = max(int(round(0.015 * fs)), 1)

    contact = sustained_index(
        force_filtered > contact_threshold,
        min_contact_samples,
        start=0,
    )
    if contact is None:
        raise ValueError("No se pudo detectar el primer contacto del DJ.")

    takeoff = sustained_index(
        force_filtered < contact_threshold,
        min_flight_samples,
        start=min(contact + int(round(0.050 * fs)), len(force_filtered) - 1),
    )

    landing = None
    if takeoff is not None:
        landing = sustained_index(
            force_filtered > contact_threshold,
            min_flight_samples,
            start=min(takeoff + int(round(0.050 * fs)), len(force_filtered) - 1),
        )

    # Velocidad vertical estimada en el primer contacto.
    initial_velocity = -math.sqrt(2.0 * G * drop_height_m)
    fall_time = math.sqrt(2.0 * drop_height_m / G)

    contact_time = float(time[contact])
    release_time = contact_time - fall_time
    release = int(np.searchsorted(time, release_time, side="left"))
    release = max(0, min(release, contact))

    n = len(time)
    sample_index = np.arange(n)

    velocity = np.full(n, np.nan, dtype=float)
    displacement_contact = np.full(n, np.nan, dtype=float)
    position_reconstructed = np.full(n, np.nan, dtype=float)

    # Antes de la liberación se representa al sujeto estático sobre el cajón.
    before_release = time < release_time
    velocity[before_release] = 0.0
    displacement_contact[before_release] = drop_height_m
    position_reconstructed[before_release] = 0.0

    # Caída balística previa al contacto.
    falling = (
        (time >= release_time)
        & (sample_index < contact)
    )
    tau = time[falling] - release_time
    velocity[falling] = -G * tau
    position_reconstructed[falling] = -0.5 * G * tau**2
    displacement_contact[falling] = (
        position_reconstructed[falling] + drop_height_m
    )

    # Desde el contacto: integración de la aceleración derivada de la fuerza.
    velocity[contact:] = initial_velocity + cumulative_trapezoid(
        acceleration[contact:],
        time[contact:],
        initial=0.0,
    )
    displacement_contact[contact:] = cumulative_trapezoid(
        velocity[contact:],
        time[contact:],
        initial=0.0,
    )
    position_reconstructed[contact:] = (
        displacement_contact[contact:] - drop_height_m
    )

    if (
        not np.isfinite(velocity).all()
        or not np.isfinite(displacement_contact).all()
        or not np.isfinite(position_reconstructed).all()
    ):
        raise ValueError(
            "La reconstrucción del DJ produjo valores no finitos. "
            "Compruebe el tiempo, la fuerza y la altura de caída."
        )

    # Punto más bajo del COM: primera transición de velocidad negativa a positiva.
    search_end = (
        int(takeoff)
        if takeoff is not None
        else min(contact + int(round(1.5 * fs)), n - 1)
    )
    velocity_contact = velocity[contact:search_end + 1]
    crossings = np.where(
        (velocity_contact[:-1] < 0)
        & (velocity_contact[1:] >= 0)
    )[0]

    if len(crossings):
        bottom = contact + int(crossings[0]) + 1
    else:
        bottom = contact + int(
            np.nanargmin(displacement_contact[contact:search_end + 1])
        )

    events = {
        "release": release,
        "onset": contact,
        "contact": contact,
        "bottom": bottom,
        "takeoff": takeoff,
        "landing": landing,
        "quiet_start": None,
        "quiet_end": None,
        "movement_threshold_N": np.nan,
        "contact_threshold_N": contact_threshold,
        "drop_height_m": float(drop_height_m),
        "initial_velocity_m_s": float(initial_velocity),
        "release_time_s": float(release_time),
    }

    return (
        force_filtered,
        acceleration,
        velocity,
        displacement_contact,
        position_reconstructed,
        events,
        inverted,
    )


def build_phase_labels(n_samples, events, mode):
    """Devuelve una etiqueta biomecánica de fase para cada muestra."""
    labels = np.full(int(n_samples), "sin_clasificar", dtype=object)

    takeoff = events.get("takeoff")
    landing = events.get("landing")

    if mode == "dj":
        release = events.get("release")
        contact = events.get("contact")
        bottom = events.get("bottom")

        labels[:] = "antes_liberacion"

        if release is not None:
            labels[int(release):] = "caida_previa"
        if contact is not None:
            labels[int(contact):] = "fase_excentrica"
        if bottom is not None:
            labels[int(bottom):] = "fase_concentrica"
    else:
        onset = events.get("onset")
        labels[:] = "reposo"
        if onset is not None:
            labels[int(onset):] = "fase_propulsiva"

    if takeoff is not None:
        labels[int(takeoff):] = "vuelo"
    if landing is not None:
        labels[int(landing):] = "post_aterrizaje"

    return labels

def infer_jump_mode(metadata):
    text = " ".join(
        str(metadata.get(k, ""))
        for k in ["tipo_salto", "archivo", "id_archivo"]
    ).upper()
    return "dj" if re.search(r"(^|[^A-Z])DJ([^A-Z]|$)", text) else "quiet"



def compute_metrics(processed, events, bw, mode):
    """
    Calcula las métricas generales y específicas del salto.

    La altura se obtiene mediante tres métodos:
    1) velocidad vertical del COM en el despegue;
    2) desplazamiento del COM entre el despegue y el ápice;
    3) tiempo de vuelo, cuando se detecta el aterrizaje.
    """
    time = processed["tiempo_s"].to_numpy(dtype=float)
    f_bw = processed["fuerza_BW"].to_numpy(dtype=float)
    velocity = processed["velocidad_COM_m_s"].to_numpy(dtype=float)
    primary_displacement = processed[
        "desplazamiento_COM_m"
    ].to_numpy(dtype=float)

    start = events.get("contact") if mode == "dj" else events.get("onset")
    if start is None:
        start = 0

    takeoff = events.get("takeoff")
    landing = events.get("landing")
    end = landing if landing is not None else len(processed) - 1

    metrics = {
        "peso_corporal_N": float(bw),
        "masa_corporal_kg": float(bw / G),
        "fuerza_maxima_BW": float(np.nanmax(f_bw[start:end + 1])),
        "velocidad_minima_m_s": float(
            np.nanmin(velocity[start:end + 1])
        ),
        "velocidad_maxima_m_s": float(
            np.nanmax(velocity[start:end + 1])
        ),
        "desplazamiento_maximo_m": float(
            np.nanmax(primary_displacement[start:end + 1])
        ),
    }

    # ------------------------------------------------------------------
    # Altura a partir de la velocidad de despegue
    # h = v_TO² / (2g)
    # ------------------------------------------------------------------
    if takeoff is not None:
        takeoff_velocity = float(velocity[takeoff])
        metrics["velocidad_despegue_m_s"] = takeoff_velocity
        metrics["altura_velocidad_despegue_m"] = float(
            max(takeoff_velocity, 0.0) ** 2 / (2.0 * G)
        )

        # Alias conservado para compatibilidad con versiones anteriores.
        metrics["altura_impulso_m"] = metrics[
            "altura_velocidad_despegue_m"
        ]

    # ------------------------------------------------------------------
    # Altura a partir del desplazamiento del COM
    # h = z_apex - z_takeoff
    # ------------------------------------------------------------------
    if takeoff is not None and takeoff < len(processed) - 1:
        search_end = (
            int(landing)
            if landing is not None and landing > takeoff
            else len(processed) - 1
        )

        # Primero se intenta localizar el ápice como la primera transición
        # de velocidad positiva a velocidad no positiva durante el vuelo.
        flight_velocity = velocity[takeoff:search_end + 1]
        apex_crossings = np.where(
            (flight_velocity[:-1] > 0)
            & (flight_velocity[1:] <= 0)
        )[0]

        if len(apex_crossings):
            apex = takeoff + int(apex_crossings[0]) + 1
        else:
            # Respaldo: máximo desplazamiento dentro de la ventana de vuelo.
            flight_displacement = primary_displacement[
                takeoff:search_end + 1
            ]
            apex = takeoff + int(np.nanargmax(flight_displacement))

        events["apex"] = int(apex)

        displacement_height = float(
            primary_displacement[apex]
            - primary_displacement[takeoff]
        )
        metrics["altura_desplazamiento_COM_m"] = float(
            max(displacement_height, 0.0)
        )
        metrics["posicion_COM_despegue_m"] = float(
            primary_displacement[takeoff]
        )
        metrics["posicion_COM_apice_m"] = float(
            primary_displacement[apex]
        )
        metrics["tiempo_despegue_apice_s"] = float(
            time[apex] - time[takeoff]
        )

        if "altura_velocidad_despegue_m" in metrics:
            difference = float(
                metrics["altura_desplazamiento_COM_m"]
                - metrics["altura_velocidad_despegue_m"]
            )
            metrics[
                "diferencia_altura_COM_menos_velocidad_m"
            ] = difference
            metrics[
                "diferencia_absoluta_alturas_m"
            ] = abs(difference)

            mean_height = np.mean([
                metrics["altura_desplazamiento_COM_m"],
                metrics["altura_velocidad_despegue_m"],
            ])
            metrics[
                "diferencia_relativa_alturas_pct"
            ] = float(
                100.0 * abs(difference) / mean_height
                if mean_height > 0
                else np.nan
            )

    # ------------------------------------------------------------------
    # Altura mediante tiempo de vuelo
    # ------------------------------------------------------------------
    if takeoff is not None and landing is not None:
        flight_time = float(time[landing] - time[takeoff])
        metrics["tiempo_vuelo_s"] = flight_time
        metrics["altura_tiempo_vuelo_m"] = float(
            G * flight_time**2 / 8.0
        )

    # ------------------------------------------------------------------
    # Métricas específicas del drop jump
    # ------------------------------------------------------------------
    if mode == "dj":
        contact = events.get("contact")
        bottom = events.get("bottom")

        relative_displacement = processed[
            "desplazamiento_COM_rel_contacto_m"
        ].to_numpy(dtype=float)

        if contact is not None:
            metrics["velocidad_contacto_m_s"] = float(
                velocity[contact]
            )

        if (
            contact is not None
            and takeoff is not None
            and takeoff > contact
        ):
            contact_time = float(time[takeoff] - time[contact])
            metrics["tiempo_contacto_s"] = contact_time

            contact_segment = relative_displacement[
                contact:takeoff + 1
            ]
            minimum_relative = float(np.nanmin(contact_segment))

            metrics[
                "desplazamiento_minimo_rel_contacto_m"
            ] = minimum_relative
            metrics["descenso_excentrico_m"] = float(
                max(-minimum_relative, 0.0)
            )
            metrics[
                "desplazamiento_despegue_rel_contacto_m"
            ] = float(relative_displacement[takeoff])
            metrics[
                "rango_desplazamiento_contacto_m"
            ] = float(
                np.nanmax(contact_segment)
                - np.nanmin(contact_segment)
            )

            if bottom is not None and contact <= bottom <= takeoff:
                metrics["tiempo_fase_excentrica_s"] = float(
                    time[bottom] - time[contact]
                )
                metrics["tiempo_fase_concentrica_s"] = float(
                    time[takeoff] - time[bottom]
                )
                metrics["ascenso_concentrico_m"] = float(
                    relative_displacement[takeoff]
                    - relative_displacement[bottom]
                )

            if (
                "altura_tiempo_vuelo_m" in metrics
                and contact_time > 0
            ):
                metrics["RSI_m_s"] = float(
                    metrics["altura_tiempo_vuelo_m"]
                    / contact_time
                )

            # RSI-modificado usando los otros dos métodos de altura.
            if (
                "altura_velocidad_despegue_m" in metrics
                and contact_time > 0
            ):
                metrics["RSI_velocidad_m_s"] = float(
                    metrics["altura_velocidad_despegue_m"]
                    / contact_time
                )

            if (
                "altura_desplazamiento_COM_m" in metrics
                and contact_time > 0
            ):
                metrics["RSI_desplazamiento_COM_m_s"] = float(
                    metrics["altura_desplazamiento_COM_m"]
                    / contact_time
                )

    return metrics


def event_times(processed, events):
    time = processed["tiempo_s"].to_numpy(dtype=float)
    result = {}

    for key in [
        "quiet_start",
        "quiet_end",
        "release",
        "onset",
        "contact",
        "bottom",
        "takeoff",
        "apex",
        "landing",
    ]:
        index = events.get(key)
        result[key] = (
            float(time[index])
            if isinstance(index, (int, np.integer))
            and 0 <= index < len(time)
            else None
        )

    # El instante de liberación puede caer entre dos muestras.
    if events.get("release_time_s") is not None:
        result["release"] = float(events["release_time_s"])

    return result



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 35.2 MB/s eta 0:00:00


In [ ]:
#@title 2. Subir el TXT individual o la matriz CSV
# IMPORTANTE: la carga nativa de Colab debe ejecutarse directamente en esta celda.
# No se introduce dentro de widgets.Output(), porque eso puede provocar el error
# "google.colab._files is undefined".

def process_source_path(path, name):
    extension = Path(name).suffix.lower()
    if extension not in {".txt", ".csv"}:
        raise ValueError("Seleccione un archivo con extensión .txt o .csv.")

    source_kind = "txt" if extension == ".txt" else "csv"

    STATE.update({
        "source_kind": source_kind,
        "source_path": str(path),
        "source_name": name,
        "catalog": None,
        "selected_id": None,
        "selected_metadata": {},
        "selected_trial": None,
        "processed": None,
        "events": None,
        "metrics": None,
    })

    if source_kind == "txt":
        print("Leyendo el TXT y comprobando sus canales...")
        trial, metadata, raw_header = parse_kistler_txt(path)
        STATE["selected_trial"] = trial
        STATE["selected_metadata"] = metadata
        STATE["txt_metadata"] = raw_header

        display(HTML(
            f"<div style='padding:10px;border-left:5px solid #2e7d32;'>"
            f"<b>TXT cargado correctamente:</b> {name}<br>"
            f"{metadata['n_muestras']:,} muestras · "
            f"{metadata['frecuencia_hz']:.1f} Hz · "
            f"{metadata['n_plataformas']} plataforma(s) · "
            f"{metadata['duracion_s']:.3f} s"
            f"</div>"
        ))
    else:
        print("Leyendo el catálogo de la matriz. No se carga toda en memoria...")
        catalog, sep = build_csv_catalog(path)
        STATE["catalog"] = catalog

        display(HTML(
            f"<div style='padding:10px;border-left:5px solid #2e7d32;'>"
            f"<b>Matriz cargada correctamente:</b> {name}<br>"
            f"{len(catalog):,} saltos identificados · separador detectado: "
            f"<code>{repr(sep)}</code>"
            f"</div>"
        ))

    print("\n✅ Ahora ejecute la celda 3 para seleccionar o verificar el salto.")


# Reinicia únicamente los mensajes de esta celda, sin encapsular el cargador.
clear_output(wait=True)

if EN_COLAB:
    try:
        print(
            "Seleccione UN archivo TXT o CSV en el cuadro que aparecerá debajo.\n"
            "Para que el selector funcione, ejecute esta celda manualmente y no desde "
            "una salida guardada de una sesión anterior."
        )

        # Debe permanecer en el nivel principal de la celda de Colab.
        uploaded = files.upload()

        if len(uploaded) == 0:
            raise ValueError("No se ha seleccionado ningún archivo.")
        if len(uploaded) != 1:
            raise ValueError("Debe seleccionar exactamente un archivo.")

        name, content = next(iter(uploaded.items()))
        destination = Path("/content") / Path(name).name

        # files.upload() normalmente ya crea el archivo, pero se garantiza su escritura.
        if not destination.exists() or destination.stat().st_size == 0:
            destination.write_bytes(content)

        del uploaded
        process_source_path(str(destination), destination.name)

    except Exception as exc:
        display(HTML(
            f"<div style='padding:10px;border-left:5px solid #c62828;'>"
            f"<b>Error durante la carga:</b> {exc}<br><br>"
            f"Vuelva a ejecutar <b>solo esta celda</b>. Si Colab se ha reconectado, "
            f"ejecute antes la celda 1 para restaurar las funciones."
            f"</div>"
        ))

else:
    display(HTML(
        "<p>Este notebook está optimizado para Google Colab. "
        "En Jupyter local puede utilizar el selector siguiente.</p>"
    ))

    local_upload = widgets.FileUpload(
        accept=".txt,.csv",
        multiple=False,
        description="Elegir archivo",
        button_style="primary",
        icon="upload",
    )
    local_button = widgets.Button(
        description="Procesar archivo",
        button_style="success",
        icon="check",
    )
    local_output = widgets.Output()

    def process_local(_):
        with local_output:
            clear_output(wait=True)
            try:
                path, name = _extract_uploaded_file(local_upload)
                process_source_path(path, name)
            except Exception as exc:
                display(HTML(
                    f"<div style='padding:10px;border-left:5px solid #c62828;'>"
                    f"<b>Error:</b> {exc}</div>"
                ))

    local_button.on_click(process_local)
    display(widgets.VBox([local_upload, local_button, local_output]))


Seleccione UN archivo TXT o CSV en el cuadro que aparecerá debajo.
Para que el selector funcione, ejecute esta celda manualmente y no desde una salida guardada de una sesión anterior.


Saving A1CMJ 001.txt to A1CMJ 001.txt
Leyendo el TXT y comprobando sus canales...



✅ Ahora ejecute la celda 3 para seleccionar o verificar el salto.


In [ ]:

#@title 3. Seleccionar un salto de la matriz o verificar el TXT
salida_selector = widgets.Output()
boton_actualizar_selector = widgets.Button(
    description="Actualizar selector",
    button_style="info",
    icon="refresh",
)

def _display_trial_check(metadata, trial):
    rows = {
        "Archivo / ID": metadata.get("id_archivo") or metadata.get("archivo"),
        "Año": metadata.get("anio", ""),
        "Grupo": metadata.get("grupo", ""),
        "Sujeto": metadata.get("codigo_sujeto", ""),
        "Tipo de salto": metadata.get("tipo_salto", ""),
        "Intento": metadata.get("intento", ""),
        "Frecuencia": f"{metadata.get('frecuencia_hz', np.nan):.1f} Hz",
        "Muestras": f"{len(trial):,}",
        "Duración": f"{metadata.get('duracion_s', np.nan):.3f} s",
        "Fz mínima": f"{trial['Fz_total'].min():.2f} N",
        "Fz máxima": f"{trial['Fz_total'].max():.2f} N",
    }
    table = "".join(
        f"<tr><th style='text-align:left;padding:4px 12px 4px 0'>{k}</th>"
        f"<td>{v}</td></tr>"
        for k, v in rows.items()
        if v not in ["", None]
    )
    display(HTML(
        "<div style='padding:10px;border:1px solid #ccc;border-radius:6px'>"
        "<b>Verificación del salto seleccionado</b>"
        f"<table style='margin-top:8px'>{table}</table></div>"
    ))

def construir_selector(_=None):
    with salida_selector:
        clear_output()

        if STATE["source_kind"] is None:
            print("⚠️ Primero cargue y procese un archivo en la celda 2.")
            return

        if STATE["source_kind"] == "txt":
            trial = STATE["selected_trial"]
            metadata = get_trial_metadata(trial, STATE["selected_metadata"])
            STATE["selected_metadata"] = metadata
            _display_trial_check(metadata, trial)
            display(HTML(
                "<p style='color:#2e7d32'><b>✅ El TXT individual queda seleccionado "
                "automáticamente.</b> Continúe con la celda 4.</p>"
            ))
            return

        catalog = STATE["catalog"].copy()

        def values_for(column):
            values = sorted(
                {str(v) for v in catalog[column].dropna().tolist() if str(v) not in {"", "nan"}}
            )
            return ["Todos"] + values

        filtro_anio = widgets.Dropdown(
            options=values_for("anio"), description="Año:",
            style={"description_width": "initial"}
        )
        filtro_grupo = widgets.Dropdown(
            options=values_for("grupo"), description="Grupo:",
            style={"description_width": "initial"}
        )
        filtro_tipo = widgets.Dropdown(
            options=values_for("tipo_salto"), description="Salto:",
            style={"description_width": "initial"}
        )
        filtro_sujeto = widgets.Dropdown(
            options=values_for("codigo_sujeto"), description="Sujeto:",
            style={"description_width": "initial"}
        )

        selector_salto = widgets.Dropdown(
            options=[],
            description="Archivo:",
            layout=widgets.Layout(width="98%"),
            style={"description_width": "initial"},
        )
        boton_cargar_salto = widgets.Button(
            description="Cargar salto seleccionado",
            button_style="success",
            icon="check",
        )
        salida_carga = widgets.Output()

        def filtered_catalog():
            selected = catalog.copy()
            for widget, column in [
                (filtro_anio, "anio"),
                (filtro_grupo, "grupo"),
                (filtro_tipo, "tipo_salto"),
                (filtro_sujeto, "codigo_sujeto"),
            ]:
                if widget.value != "Todos":
                    selected = selected[
                        selected[column].astype(str) == str(widget.value)
                    ]
            return selected

        def refresh_options(change=None):
            subset = filtered_catalog()
            options = []
            for _, row in subset.iterrows():
                label = (
                    f"{row.get('anio', '')} | Grupo {row.get('grupo', '')} | "
                    f"{row.get('codigo_sujeto', '')} | {row.get('tipo_salto', '')} | "
                    f"intento {row.get('intento', '')} | {row.get('archivo', '')}"
                )
                options.append((label, str(row["id_archivo_selector"])))
            selector_salto.options = options
            selector_salto.disabled = not bool(options)
            boton_cargar_salto.disabled = not bool(options)

        for w in [filtro_anio, filtro_grupo, filtro_tipo, filtro_sujeto]:
            w.observe(refresh_options, names="value")

        def cargar_salto(_):
            with salida_carga:
                clear_output()
                try:
                    if selector_salto.value is None:
                        raise ValueError("No hay ningún salto seleccionado.")

                    print("Cargando solo las filas del salto seleccionado...")
                    trial = load_csv_trial(
                        STATE["source_path"],
                        selector_salto.value
                    )
                    meta_row = catalog.loc[
                        catalog["id_archivo_selector"].astype(str)
                        == str(selector_salto.value)
                    ].iloc[0].to_dict()

                    metadata = get_trial_metadata(
                        trial,
                        {
                            **meta_row,
                            "id_archivo": selector_salto.value,
                            "archivo": meta_row.get("archivo", selector_salto.value),
                        }
                    )

                    STATE["selected_id"] = selector_salto.value
                    STATE["selected_trial"] = trial
                    STATE["selected_metadata"] = metadata
                    STATE["processed"] = None
                    STATE["events"] = None
                    STATE["metrics"] = None

                    clear_output()
                    _display_trial_check(metadata, trial)
                    display(HTML(
                        "<p style='color:#2e7d32'><b>✅ Salto cargado y verificado.</b> "
                        "Continúe con la celda 4.</p>"
                    ))
                except Exception as exc:
                    display(HTML(
                        f"<div style='padding:10px;border-left:5px solid #c62828;'>"
                        f"<b>Error:</b> {exc}</div>"
                    ))

        boton_cargar_salto.on_click(cargar_salto)
        refresh_options()

        display(HTML("<h4>Filtre la matriz y elija un archivo de salto</h4>"))
        display(widgets.HBox([filtro_anio, filtro_grupo]))
        display(widgets.HBox([filtro_tipo, filtro_sujeto]))
        display(selector_salto, boton_cargar_salto, salida_carga)

boton_actualizar_selector.on_click(construir_selector)
display(boton_actualizar_selector, salida_selector)
construir_selector()


Button(button_style='info', description='Actualizar selector', icon='refresh', style=ButtonStyle())

Output()

In [ ]:

#@title 4. Configurar y calcular fuerza, BW y cinemática del centro de masas
modo_salto = widgets.Dropdown(
    options=[
        ("Detectar automáticamente", "auto"),
        ("CMJ/SJ: comienza en reposo sobre la plataforma", "quiet"),
        ("DJ: comienza con una caída desde un cajón", "dj"),
    ],
    description="Modo:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="680px"),
)

modo_bw = widgets.Dropdown(
    options=[
        ("Automático / usar metadato del TXT", "auto"),
        ("Introducir masa corporal en kg", "mass"),
        ("Introducir peso corporal en N", "bw"),
    ],
    description="Peso corporal:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="680px"),
)

valor_manual = widgets.BoundedFloatText(
    value=70.0,
    min=0.1,
    max=3000.0,
    step=0.1,
    description="Valor manual:",
    style={"description_width": "initial"},
)

altura_caida_cm = widgets.BoundedFloatText(
    value=0.0,
    min=0.0,
    max=200.0,
    step=1.0,
    description="Caída efectiva DJ (cm):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

frecuencia_corte = widgets.BoundedFloatText(
    value=20.0,
    min=1.0,
    max=300.0,
    step=1.0,
    description="Filtro (Hz):",
    style={"description_width": "initial"},
)

boton_analizar = widgets.Button(
    description="Analizar salto",
    button_style="success",
    icon="play",
)

salida_analisis = widgets.Output()

def _fmt_metric(metrics, key, factor=1.0, decimals=3, suffix=""):
    value = metrics.get(key, np.nan)
    if not np.isfinite(value):
        return "No disponible"
    return f"{value * factor:.{decimals}f}{suffix}"

def analizar_salto(_):
    with salida_analisis:
        clear_output()

        try:
            if STATE["selected_trial"] is None:
                raise ValueError(
                    "No hay un salto seleccionado. Complete primero las celdas 2 y 3."
                )

            trial = standardise_trial(STATE["selected_trial"])
            metadata = get_trial_metadata(
                trial,
                STATE["selected_metadata"],
            )
            fs = float(metadata["frecuencia_hz"])

            raw_force, sign_inverted_for_estimation = force_sign_correct(
                trial["Fz_total"].to_numpy(dtype=float)
            )
            filtered_for_estimation = lowpass(
                raw_force,
                fs,
                frecuencia_corte.value,
            )

            chosen_mode = (
                infer_jump_mode(metadata)
                if modo_salto.value == "auto"
                else modo_salto.value
            )

            estimated_bw, quiet_idx, quiet_sd = estimate_bw_and_quiet(
                trial["tiempo_s"],
                filtered_for_estimation,
                fs,
                prefer_early=(chosen_mode == "quiet"),
            )

            metadata_bw = _safe_float(
                metadata.get("peso_corporal_metadata_N", np.nan)
            )

            if modo_bw.value == "mass":
                bw = float(valor_manual.value) * G
                bw_source = "masa corporal introducida manualmente"
            elif modo_bw.value == "bw":
                bw = float(valor_manual.value)
                bw_source = "peso corporal introducido manualmente"
            elif np.isfinite(metadata_bw) and 250 <= metadata_bw <= 2000:
                bw = float(metadata_bw)
                bw_source = "metadato «Normalized force (N)» del TXT"
            else:
                bw = float(estimated_bw)
                bw_source = "periodo estable detectado automáticamente"

            if not 200 <= bw <= 2500:
                raise ValueError(
                    f"El peso corporal utilizado ({bw:.1f} N) no parece plausible."
                )

            reconstructed_position = None
            relative_displacement = None

            if chosen_mode == "quiet":
                (
                    force_filtered,
                    acceleration,
                    velocity,
                    displacement,
                    events,
                    inverted,
                ) = analyse_quiet_start(
                    trial,
                    fs,
                    bw,
                    frecuencia_corte.value,
                    quiet_idx,
                )

                primary_displacement = displacement
                relative_displacement = displacement

            else:
                (
                    force_filtered,
                    acceleration,
                    velocity,
                    relative_displacement,
                    reconstructed_position,
                    events,
                    inverted,
                ) = analyse_drop_jump(
                    trial,
                    fs,
                    bw,
                    frecuencia_corte.value,
                    altura_caida_cm.value / 100.0,
                )

                # En DJ, la variable principal mostrada es la trayectoria completa
                # reconstruida desde la liberación. La señal relativa al contacto
                # se conserva en una columna independiente.
                primary_displacement = reconstructed_position

            processed = pd.DataFrame({
                "tiempo_s": trial["tiempo_s"].to_numpy(dtype=float),
                "fuerza_vertical_N": force_filtered,
                "fuerza_BW": force_filtered / bw,
                "aceleracion_COM_m_s2": acceleration,
                "velocidad_COM_m_s": velocity,
                "desplazamiento_COM_m": primary_displacement,
                "fase_salto": build_phase_labels(
                    len(trial),
                    events,
                    chosen_mode,
                ),
            })

            if chosen_mode == "dj":
                processed["desplazamiento_COM_rel_contacto_m"] = (
                    relative_displacement
                )
                processed["posicion_COM_reconstruida_desde_liberacion_m"] = (
                    reconstructed_position
                )
                processed["altura_caida_introducida_m"] = (
                    altura_caida_cm.value / 100.0
                )
            else:
                processed["desplazamiento_COM_rel_inicio_m"] = (
                    relative_displacement
                )

            # Alias conservados para compatibilidad con versiones anteriores.
            processed["aceleracion_m_s2"] = processed["aceleracion_COM_m_s2"]
            processed["velocidad_m_s"] = processed["velocidad_COM_m_s"]
            processed["desplazamiento_m"] = processed["desplazamiento_COM_m"]

            required_columns = [
                "aceleracion_COM_m_s2",
                "velocidad_COM_m_s",
                "desplazamiento_COM_m",
            ]
            for column in required_columns:
                if not np.isfinite(
                    processed[column].to_numpy(dtype=float)
                ).all():
                    raise ValueError(
                        f"La columna «{column}» contiene valores no finitos."
                    )

            metrics = compute_metrics(
                processed,
                events,
                bw,
                chosen_mode,
            )

            # Señal relativa al despegue para comprobar visualmente la altura
            # calculada mediante el desplazamiento del COM.
            takeoff_index = events.get("takeoff")
            if takeoff_index is not None:
                takeoff_position = float(
                    processed["desplazamiento_COM_m"].iloc[takeoff_index]
                )
                processed[
                    "desplazamiento_COM_desde_despegue_m"
                ] = (
                    processed["desplazamiento_COM_m"]
                    - takeoff_position
                )
            else:
                processed[
                    "desplazamiento_COM_desde_despegue_m"
                ] = np.nan

            # Las alturas globales se repiten como columnas para que permanezcan
            # disponibles en el CSV de la serie temporal.
            processed[
                "altura_salto_velocidad_despegue_m"
            ] = metrics.get(
                "altura_velocidad_despegue_m",
                np.nan,
            )
            processed[
                "altura_salto_desplazamiento_COM_m"
            ] = metrics.get(
                "altura_desplazamiento_COM_m",
                np.nan,
            )
            processed[
                "altura_salto_tiempo_vuelo_m"
            ] = metrics.get(
                "altura_tiempo_vuelo_m",
                np.nan,
            )
            processed[
                "diferencia_altura_COM_menos_velocidad_m"
            ] = metrics.get(
                "diferencia_altura_COM_menos_velocidad_m",
                np.nan,
            )

            STATE["selected_trial"] = trial
            STATE["selected_metadata"] = metadata
            STATE["processed"] = processed
            STATE["events"] = events
            STATE["metrics"] = metrics
            STATE["analysis_mode"] = chosen_mode
            STATE["bw_N"] = bw
            STATE["drop_height_m"] = (
                altura_caida_cm.value / 100.0
                if chosen_mode == "dj"
                else None
            )

            times = event_times(processed, events)

            summary = {
                "Modo aplicado": (
                    "DJ: caída balística + integración desde el contacto"
                    if chosen_mode == "dj"
                    else "CMJ/SJ con velocidad inicial igual a 0"
                ),
                "Origen del BW": bw_source,
                "Peso corporal": f"{bw:.2f} N",
                "Masa corporal": f"{bw / G:.2f} kg",
                "Periodo estable usado para BW": (
                    f"{processed['tiempo_s'].iloc[quiet_idx[0]]:.3f}–"
                    f"{processed['tiempo_s'].iloc[quiet_idx[1] - 1]:.3f} s"
                ),
                "Inicio / contacto": (
                    f"{(
                        times['contact']
                        if chosen_mode == 'dj'
                        else times['onset']
                    ):.3f} s"
                ),
                "Despegue": (
                    f"{times['takeoff']:.3f} s"
                    if times["takeoff"] is not None
                    else "No detectado"
                ),
                "Aterrizaje": (
                    f"{times['landing']:.3f} s"
                    if times["landing"] is not None
                    else "No detectado"
                ),
                "Fuerza máxima": (
                    f"{metrics['fuerza_maxima_BW']:.2f} BW"
                ),
                "Velocidad de despegue": _fmt_metric(
                    metrics,
                    "velocidad_despegue_m_s",
                    decimals=3,
                    suffix=" m·s⁻¹",
                ),
                "Altura por velocidad de despegue": _fmt_metric(
                    metrics,
                    "altura_velocidad_despegue_m",
                    factor=100.0,
                    decimals=1,
                    suffix=" cm",
                ),
                "Ápice del vuelo": (
                    f"{times['apex']:.3f} s"
                    if times.get("apex") is not None
                    else "No detectado"
                ),
                "Altura por desplazamiento del COM": _fmt_metric(
                    metrics,
                    "altura_desplazamiento_COM_m",
                    factor=100.0,
                    decimals=1,
                    suffix=" cm",
                ),
                "Diferencia COM − velocidad": _fmt_metric(
                    metrics,
                    "diferencia_altura_COM_menos_velocidad_m",
                    factor=100.0,
                    decimals=1,
                    suffix=" cm",
                ),
                "Diferencia relativa entre métodos": _fmt_metric(
                    metrics,
                    "diferencia_relativa_alturas_pct",
                    decimals=1,
                    suffix=" %",
                ),
                "Tiempo de vuelo": _fmt_metric(
                    metrics,
                    "tiempo_vuelo_s",
                    decimals=3,
                    suffix=" s",
                ),
                "Altura por tiempo de vuelo": _fmt_metric(
                    metrics,
                    "altura_tiempo_vuelo_m",
                    factor=100.0,
                    decimals=1,
                    suffix=" cm",
                ),
            }

            if chosen_mode == "dj":
                summary.update({
                    "Liberación reconstruida": (
                        f"{times['release']:.3f} s"
                        if times["release"] is not None
                        else "Fuera del registro"
                    ),
                    "Altura usada en la reconstrucción": (
                        f"{altura_caida_cm.value:.1f} cm"
                    ),
                    "Velocidad estimada en contacto": _fmt_metric(
                        metrics,
                        "velocidad_contacto_m_s",
                        decimals=3,
                        suffix=" m·s⁻¹",
                    ),
                    "Punto más bajo del COM": (
                        f"{times['bottom']:.3f} s"
                        if times["bottom"] is not None
                        else "No detectado"
                    ),
                    "Tiempo de contacto": _fmt_metric(
                        metrics,
                        "tiempo_contacto_s",
                        decimals=3,
                        suffix=" s",
                    ),
                    "Descenso excéntrico tras el contacto": _fmt_metric(
                        metrics,
                        "descenso_excentrico_m",
                        factor=100.0,
                        decimals=1,
                        suffix=" cm",
                    ),
                    "Ascenso concéntrico": _fmt_metric(
                        metrics,
                        "ascenso_concentrico_m",
                        factor=100.0,
                        decimals=1,
                        suffix=" cm",
                    ),
                    "Posición mínima respecto al contacto": _fmt_metric(
                        metrics,
                        "desplazamiento_minimo_rel_contacto_m",
                        factor=100.0,
                        decimals=1,
                        suffix=" cm",
                    ),
                    "Posición en despegue respecto al contacto": _fmt_metric(
                        metrics,
                        "desplazamiento_despegue_rel_contacto_m",
                        factor=100.0,
                        decimals=1,
                        suffix=" cm",
                    ),
                    "RSI": _fmt_metric(
                        metrics,
                        "RSI_m_s",
                        decimals=3,
                        suffix=" m·s⁻¹",
                    ),
                })

            summary.update({
                "Muestras con velocidad calculada": (
                    f"{processed['velocidad_COM_m_s'].notna().sum():,}"
                    f" / {len(processed):,}"
                ),
                "Muestras con desplazamiento calculado": (
                    f"{processed['desplazamiento_COM_m'].notna().sum():,}"
                    f" / {len(processed):,}"
                ),
            })

            rows = "".join(
                f"<tr>"
                f"<th style='text-align:left;padding:4px 15px 4px 0'>{key}</th>"
                f"<td>{value}</td>"
                f"</tr>"
                for key, value in summary.items()
            )

            display(HTML(
                "<div style='padding:12px;border-left:5px solid #2e7d32;'>"
                "<b>✅ Análisis completado</b>"
                f"<table style='margin-top:8px'>{rows}</table>"
                "</div>"
            ))

            if inverted or sign_inverted_for_estimation:
                print(
                    "Nota: el signo de la fuerza vertical se invirtió automáticamente."
                )

            if chosen_mode == "dj":
                print(
                    "\nInterpretación del desplazamiento DJ:"
                    "\n• «desplazamiento_COM_rel_contacto_m» vale 0 m en el contacto "
                    "y cuantifica el descenso excéntrico y el ascenso posterior."
                    "\n• «posicion_COM_reconstruida_desde_liberacion_m» vale 0 m "
                    "en la liberación y añade la caída balística estimada."
                    "\n• La altura del cajón es una aproximación del descenso real "
                    "del COM; una técnica de bajada activa puede modificarlo."
                )

            print(
                "\nMétodos de altura del salto:"
                "\n• Velocidad de despegue: h = v_TO² / (2g)."
                "\n• Desplazamiento del COM: posición en el ápice menos "
                "posición en el despegue."
                "\n• Tiempo de vuelo: h = g·t_vuelo² / 8, cuando se detecta "
                "el aterrizaje."
                "\nLa altura por desplazamiento del COM utiliza doble "
                "integración y puede ser más sensible a la deriva."
            )

            print(
                "\nEjecute la celda 5 para generar la gráfica interactiva."
            )

        except Exception as exc:
            display(HTML(
                "<div style='padding:10px;border-left:5px solid #c62828;'>"
                f"<b>Error:</b> {exc}"
                "</div>"
            ))

boton_analizar.on_click(analizar_salto)

display(widgets.VBox([
    widgets.HTML(
        value=(
            "<h4>Configuración</h4>"
            "<p>Para CMJ y SJ puede dejar las opciones automáticas. "
            "En un DJ introduzca la caída vertical efectiva del COM. "
            "Cuando no se dispone de cinemática externa, la altura del cajón "
            "puede utilizarse como aproximación.</p>"
            "<p><b>Importante:</b> la trayectoria completa del DJ será una "
            "reconstrucción balística antes del contacto y una integración "
            "de la fuerza después del contacto.</p>"
        )
    ),
    modo_salto,
    modo_bw,
    valor_manual,
    altura_caida_cm,
    frecuencia_corte,
    boton_analizar,
    salida_analisis,
]))


In [ ]:

#@title 5. Mostrar la gráfica superpuesta e interactiva
# Ejecute esta celda después de pulsar «Analizar salto» en la celda 4.
#
# Pulse los nombres de la leyenda para mostrar u ocultar variables.

if STATE["processed"] is None:
    raise RuntimeError(
        "Primero pulse «Analizar salto» en la celda 4 y después vuelva a ejecutar esta celda."
    )

data = STATE["processed"].copy()
events = STATE["events"]
metadata = STATE["selected_metadata"]
mode = STATE.get("analysis_mode", "quiet")

colours = [
    "#636EFA",
    "#EF553B",
    "#00CC96",
    "#AB63FA",
    "#FFA15A",
    "#19D3F3",
]

series = [
    {
        "column": "fuerza_vertical_N",
        "name": "Fuerza vertical",
        "unit": "N",
        "axis": "y",
        "colour": colours[0],
        "category": "force",
        "visible": True,
    },
    {
        "column": "fuerza_BW",
        "name": "Fuerza normalizada",
        "unit": "BW",
        "axis": "y2",
        "colour": colours[1],
        "category": "force_bw",
        "visible": True,
    },
    {
        "column": "aceleracion_COM_m_s2",
        "name": "Aceleración COM",
        "unit": "m·s⁻²",
        "axis": "y3",
        "colour": colours[2],
        "category": "kinematic",
        "visible": True,
    },
    {
        "column": "velocidad_COM_m_s",
        "name": "Velocidad COM",
        "unit": "m·s⁻¹",
        "axis": "y4",
        "colour": colours[3],
        "category": "kinematic",
        "visible": True,
    },
]

if mode == "dj":
    series.extend([
        {
            "column": "posicion_COM_reconstruida_desde_liberacion_m",
            "name": "Posición COM reconstruida desde liberación",
            "unit": "m",
            "axis": "y5",
            "colour": colours[4],
            "category": "kinematic",
            "visible": True,
        },
        {
            "column": "desplazamiento_COM_rel_contacto_m",
            "name": "Desplazamiento COM relativo al contacto",
            "unit": "m",
            "axis": "y5",
            "colour": colours[5],
            "category": "kinematic",
            "visible": "legendonly",
        },
    ])
else:
    series.append({
        "column": "desplazamiento_COM_m",
        "name": "Desplazamiento COM",
        "unit": "m",
        "axis": "y5",
        "colour": colours[4],
        "category": "kinematic",
        "visible": True,
    })

missing = [
    item["column"]
    for item in series
    if item["column"] not in data.columns
]
if missing:
    raise KeyError(
        "Faltan columnas necesarias para la gráfica: "
        + ", ".join(missing)
    )

fig = go.Figure()

for item in series:
    fig.add_trace(
        go.Scatter(
            x=data["tiempo_s"],
            y=data[item["column"]],
            mode="lines",
            name=f'{item["name"]} ({item["unit"]})',
            yaxis=item["axis"],
            line=dict(
                color=item["colour"],
                width=2,
            ),
            connectgaps=False,
            visible=item["visible"],
            hovertemplate=(
                "Tiempo: %{x:.3f} s"
                f"<br>{item['name']}: "
                "%{y:.4f} "
                f"{item['unit']}<extra></extra>"
            ),
        )
    )

if mode == "dj":
    event_definitions = [
        ("release", "Liberación"),
        ("contact", "Contacto"),
        ("bottom", "Punto más bajo"),
        ("takeoff", "Despegue"),
        ("apex", "Ápice"),
        ("landing", "Aterrizaje"),
    ]
else:
    event_definitions = [
        ("quiet_start", "Inicio estable"),
        ("quiet_end", "Fin estable"),
        ("onset", "Inicio movimiento"),
        ("takeoff", "Despegue"),
        ("apex", "Ápice"),
        ("landing", "Aterrizaje"),
    ]

event_time_dict = event_times(data, events)
used_times = set()

for key, label in event_definitions:
    event_time = event_time_dict.get(key)
    if event_time is None:
        continue

    rounded_time = round(float(event_time), 6)
    if rounded_time in used_times:
        continue
    used_times.add(rounded_time)

    fig.add_shape(
        type="line",
        x0=event_time,
        x1=event_time,
        y0=0,
        y1=1,
        xref="x",
        yref="paper",
        line=dict(
            color="rgba(60,60,60,0.65)",
            width=1,
            dash="dash",
        ),
        layer="below",
    )

    fig.add_annotation(
        x=event_time,
        y=1.015,
        xref="x",
        yref="paper",
        text=label,
        showarrow=False,
        textangle=-90,
        font=dict(
            size=10,
            color="rgba(60,60,60,0.9)",
        ),
        xanchor="left",
        yanchor="bottom",
    )

start_index = (
    events.get("release")
    if mode == "dj"
    else events.get("onset")
)
end_index = events.get("landing")

initial_range = None
if start_index is not None:
    pre_margin = 0.20 if mode == "dj" else 0.60
    x0 = max(
        float(data["tiempo_s"].iloc[0]),
        float(data["tiempo_s"].iloc[int(start_index)]) - pre_margin,
    )
    x1 = (
        min(
            float(data["tiempo_s"].iloc[-1]),
            float(data["tiempo_s"].iloc[int(end_index)]) + 0.40,
        )
        if end_index is not None
        else float(data["tiempo_s"].iloc[-1])
    )
    initial_range = [x0, x1]

title_parts = [
    str(
        metadata.get("id_archivo")
        or metadata.get("archivo")
        or "Salto"
    ),
    str(metadata.get("tipo_salto", "")),
]
title = " · ".join([
    part
    for part in title_parts
    if part and part.lower() != "nan"
])

axis_common = dict(
    showgrid=False,
    zeroline=True,
    zerolinewidth=1,
    fixedrange=False,
)

all_visibility = [True for _ in series]
forces_visibility = [
    True if item["category"] in {"force", "force_bw"} else "legendonly"
    for item in series
]
kinematic_visibility = [
    True if item["category"] == "kinematic" else "legendonly"
    for item in series
]
bw_visibility = [
    True if item["category"] == "force_bw" else "legendonly"
    for item in series
]
hidden_visibility = ["legendonly" for _ in series]

fig.update_layout(
    title=dict(
        text=title,
        x=0.5,
        xanchor="center",
    ),
    height=780,
    template="plotly_white",
    hovermode="x unified",
    dragmode="pan",
    margin=dict(
        l=135,
        r=195,
        t=155,
        b=110,
    ),
    xaxis=dict(
        title="Tiempo (s)",
        domain=[0.16, 0.76],
        range=initial_range,
        rangeslider=dict(
            visible=True,
            thickness=0.10,
        ),
        showgrid=True,
        gridcolor="rgba(180,180,180,0.25)",
    ),
    yaxis=dict(
        **axis_common,
        title=dict(
            text="Fuerza (N)",
            font=dict(color=colours[0]),
        ),
        tickfont=dict(color=colours[0]),
        side="left",
        anchor="x",
    ),
    yaxis2=dict(
        **axis_common,
        title=dict(
            text="Fuerza (BW)",
            font=dict(color=colours[1]),
        ),
        tickfont=dict(color=colours[1]),
        overlaying="y",
        side="left",
        anchor="free",
        position=0.08,
    ),
    yaxis3=dict(
        **axis_common,
        title=dict(
            text="Aceleración (m·s⁻²)",
            font=dict(color=colours[2]),
        ),
        tickfont=dict(color=colours[2]),
        overlaying="y",
        side="right",
        anchor="free",
        position=0.80,
    ),
    yaxis4=dict(
        **axis_common,
        title=dict(
            text="Velocidad (m·s⁻¹)",
            font=dict(color=colours[3]),
        ),
        tickfont=dict(color=colours[3]),
        overlaying="y",
        side="right",
        anchor="free",
        position=0.89,
    ),
    yaxis5=dict(
        **axis_common,
        title=dict(
            text=(
                "Posición / desplazamiento (m)"
                if mode == "dj"
                else "Desplazamiento (m)"
            ),
            font=dict(color=colours[4]),
        ),
        tickfont=dict(color=colours[4]),
        overlaying="y",
        side="right",
        anchor="free",
        position=0.98,
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.10,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="rgba(160,160,160,0.35)",
        borderwidth=1,
        itemclick="toggle",
        itemdoubleclick="toggleothers",
    ),
    updatemenus=[{
        "type": "buttons",
        "direction": "right",
        "x": 0.16,
        "y": 1.20,
        "xanchor": "left",
        "yanchor": "bottom",
        "showactive": False,
        "pad": {"r": 6, "t": 3},
        "buttons": [
            {
                "label": "Todas",
                "method": "restyle",
                "args": [{"visible": all_visibility}],
            },
            {
                "label": "Solo fuerzas",
                "method": "restyle",
                "args": [{"visible": forces_visibility}],
            },
            {
                "label": "Solo cinemática",
                "method": "restyle",
                "args": [{"visible": kinematic_visibility}],
            },
            {
                "label": "Solo BW",
                "method": "restyle",
                "args": [{"visible": bw_visibility}],
            },
            {
                "label": "Ocultar todas",
                "method": "restyle",
                "args": [{"visible": hidden_visibility}],
            },
        ],
    }],
)

annotation_text = (
    "<b>DJ:</b> naranja = trayectoria reconstruida desde la liberación; "
    "azul claro = desplazamiento integrado relativo al contacto."
    if mode == "dj"
    else
    "<b>Selección:</b> un clic muestra u oculta una variable; "
    "doble clic deja visible solo esa variable."
)

fig.add_annotation(
    x=0.16,
    y=-0.19,
    xref="paper",
    yref="paper",
    text=annotation_text,
    showarrow=False,
    xanchor="left",
    align="left",
    font=dict(
        size=11,
        color="rgba(70,70,70,0.9)",
    ),
)

output_dir = Path(
    "/content"
    if Path("/content").exists()
    else "."
)
html_path = output_dir / "grafica_salto_superpuesta_interactiva.html"

plot_config = {
    "scrollZoom": True,
    "displaylogo": False,
    "responsive": True,
    "modeBarButtonsToAdd": [
        "drawline",
        "eraseshape",
    ],
}

fig.write_html(
    str(html_path),
    include_plotlyjs=True,
    full_html=True,
    config=plot_config,
)

fig.show(
    renderer="colab" if EN_COLAB else None,
    config=plot_config,
)

print("Controles disponibles:")
print("• Clic en una variable: mostrar u ocultar.")
print("• Doble clic: mostrar únicamente esa variable.")
print("• Botones superiores: selección rápida.")
print("• Arrastrar: desplazarse; rueda: ampliar o reducir.")
if mode == "dj":
    print(
        "• La trayectoria reconstruida incluye la caída previa estimada; "
        "el desplazamiento relativo al contacto aparece inicialmente oculto."
    )
print(f"• Copia HTML creada: {html_path.name}")


Controles disponibles:
• Clic en una variable: mostrar u ocultar.
• Doble clic: mostrar únicamente esa variable.
• Botones superiores: selección rápida.
• Arrastrar: desplazarse; rueda: ampliar o reducir.
• Copia HTML creada: grafica_salto_superpuesta_interactiva.html


In [ ]:

#@title 6. Descargar la serie procesada y el resumen de métricas
if STATE["processed"] is None:
    raise RuntimeError("Primero complete el análisis en la celda 4.")

import zipfile

base = Path(
    str(
        STATE["selected_metadata"].get("archivo")
        or STATE["selected_metadata"].get("id_archivo")
        or "salto"
    )
).stem
safe_base = re.sub(
    r"[^A-Za-z0-9_-]+",
    "_",
    base,
).strip("_")

output_dir = Path(
    "/content"
    if Path("/content").exists()
    else "."
)

series_path = output_dir / f"{safe_base}_serie_procesada.csv"
summary_path = output_dir / f"{safe_base}_resumen_metricas.csv"
zip_path = output_dir / f"{safe_base}_resultados.zip"

# Orden preferente de las variables; las columnas adicionales se conservan.
preferred_columns = [
    "tiempo_s",
    "fase_salto",
    "fuerza_vertical_N",
    "fuerza_BW",
    "aceleracion_COM_m_s2",
    "velocidad_COM_m_s",
    "desplazamiento_COM_m",
    "desplazamiento_COM_desde_despegue_m",
    "desplazamiento_COM_rel_inicio_m",
    "desplazamiento_COM_rel_contacto_m",
    "posicion_COM_reconstruida_desde_liberacion_m",
    "altura_salto_velocidad_despegue_m",
    "altura_salto_desplazamiento_COM_m",
    "altura_salto_tiempo_vuelo_m",
    "diferencia_altura_COM_menos_velocidad_m",
    "altura_caida_introducida_m",
    "aceleracion_m_s2",
    "velocidad_m_s",
    "desplazamiento_m",
]

ordered_columns = [
    column
    for column in preferred_columns
    if column in STATE["processed"].columns
]
remaining_columns = [
    column
    for column in STATE["processed"].columns
    if column not in ordered_columns
]
all_columns = ordered_columns + remaining_columns

STATE["processed"].loc[:, all_columns].to_csv(
    series_path,
    index=False,
    sep=";",
    decimal=".",
    encoding="utf-8-sig",
    float_format="%.10f",
)

# Resumen de una fila con identificación, método y todas las métricas.
summary = {
    "archivo": STATE["selected_metadata"].get(
        "archivo",
        STATE["selected_metadata"].get("id_archivo", ""),
    ),
    "id_archivo": STATE["selected_metadata"].get("id_archivo", ""),
    "tipo_salto": STATE["selected_metadata"].get("tipo_salto", ""),
    "modo_analisis": STATE.get("analysis_mode", ""),
    "frecuencia_hz": STATE["selected_metadata"].get(
        "frecuencia_hz",
        np.nan,
    ),
}

summary.update(STATE.get("metrics") or {})

pd.DataFrame([summary]).to_csv(
    summary_path,
    index=False,
    sep=";",
    decimal=".",
    encoding="utf-8-sig",
    float_format="%.10f",
)

# Un único ZIP evita que el navegador bloquee una segunda descarga.
with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(series_path, arcname=series_path.name)
    archive.write(summary_path, arcname=summary_path.name)

    html_path = output_dir / "grafica_salto_superpuesta_interactiva.html"
    if html_path.exists():
        archive.write(html_path, arcname=f"{safe_base}_grafica.html")

n_vel = int(
    STATE["processed"]["velocidad_COM_m_s"].notna().sum()
)
n_disp = int(
    STATE["processed"]["desplazamiento_COM_m"].notna().sum()
)

print(f"Serie temporal: {series_path.name}")
print(f"Resumen de métricas: {summary_path.name}")
print(f"Paquete de resultados: {zip_path.name}")
print(
    f"Velocidad calculada: "
    f"{n_vel:,}/{len(STATE['processed']):,} muestras"
)
print(
    f"Desplazamiento calculado: "
    f"{n_disp:,}/{len(STATE['processed']):,} muestras"
)

metrics = STATE.get("metrics") or {}

if np.isfinite(
    metrics.get("altura_velocidad_despegue_m", np.nan)
):
    print(
        "Altura por velocidad de despegue: "
        f"{100 * metrics['altura_velocidad_despegue_m']:.1f} cm"
    )

if np.isfinite(
    metrics.get("altura_desplazamiento_COM_m", np.nan)
):
    print(
        "Altura por desplazamiento del COM: "
        f"{100 * metrics['altura_desplazamiento_COM_m']:.1f} cm"
    )

if EN_COLAB:
    files.download(str(zip_path))
else:
    print(zip_path.resolve())


Serie temporal: A1CMJ_001_serie_procesada.csv
Resumen de métricas: A1CMJ_001_resumen_metricas.csv
Paquete de resultados: A1CMJ_001_resultados.zip
Velocidad calculada: 8,000/8,000 muestras
Desplazamiento calculado: 8,000/8,000 muestras
Altura por velocidad de despegue: 35.0 cm
Altura por desplazamiento del COM: 35.1 cm


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Consideraciones metodológicas

### 1. Fuerza vertical y peso corporal

La fuerza vertical total se obtiene sumando la componente vertical de la fuerza, `Fz`, registrada por todas las plataformas disponibles.

El peso corporal se obtiene del campo **Normalized force (N)** cuando se analiza un archivo TXT y dicho valor está disponible.

Cuando se analiza una matriz de datos, el peso corporal se estima a partir de un periodo estable de apoyo sobre la plataforma.

La masa corporal se calcula mediante:

$$
m=\frac{BW}{g}
$$

donde:

* `m` es la masa corporal, expresada en kg;
* `BW` es el peso corporal, expresado en N;
* `g` es la aceleración de la gravedad, establecida en 9.80665 m·s⁻².

---

### 2. Fuerza normalizada respecto al peso corporal

La fuerza vertical se normaliza respecto al peso corporal mediante:

$$
F_{BW}(t)=\frac{F_z(t)}{BW}
$$

Un valor de 1 BW representa una fuerza vertical igual al peso corporal.

Un valor superior a 1 BW indica que la fuerza vertical ejercida sobre la plataforma es superior al peso corporal.

Un valor inferior a 1 BW indica que la fuerza vertical es inferior al peso corporal.

---

### 3. Aceleración vertical del centro de masas

La fuerza vertical neta se calcula restando el peso corporal a la fuerza vertical registrada:

$$
F_{neta}(t)=F_z(t)-BW
$$

La aceleración vertical del centro de masas se calcula mediante:

$$
a(t)=\frac{F_z(t)-BW}{m}
$$

La misma expresión puede formularse utilizando la fuerza normalizada:

$$
a(t)=g\left(\frac{F_z(t)}{BW}-1\right)
$$

donde:

* `a(t)` es la aceleración vertical del centro de masas;
* `Fz(t)` es la fuerza vertical total en cada instante;
* `BW` es el peso corporal;
* `m` es la masa corporal;
* `g` es la aceleración de la gravedad.

Una aceleración positiva representa una aceleración dirigida hacia arriba.

Una aceleración negativa representa una aceleración dirigida hacia abajo.

---

### 4. Velocidad vertical del centro de masas

La velocidad vertical del centro de masas se obtiene integrando la aceleración respecto al tiempo:

$$
v(t)=v_0+\int_{t_0}^{t}a(\tau),d\tau
$$

donde:

* `v(t)` es la velocidad vertical del centro de masas;
* `v0` es la velocidad inicial;
* `t0` es el instante inicial de la integración;
* `a(t)` es la aceleración vertical.

La integración numérica se realiza mediante el método trapezoidal.

---

### 5. Desplazamiento vertical del centro de masas

El desplazamiento vertical del centro de masas se obtiene integrando la velocidad respecto al tiempo:

$$
z(t)=z_0+\int_{t_0}^{t}v(\tau),d\tau
$$

donde:

* `z(t)` es la posición o el desplazamiento vertical del centro de masas;
* `z0` es la posición inicial;
* `v(t)` es la velocidad vertical.

El cálculo del desplazamiento requiere una doble integración de la señal de fuerza:

1. La fuerza se transforma en aceleración.
2. La aceleración se integra para obtener velocidad.
3. La velocidad se integra para obtener desplazamiento.

---

### 6. Condiciones iniciales para CMJ y SJ

En los saltos **countermovement jump (CMJ)** y **squat jump (SJ)** se asume que el participante comienza en reposo sobre la plataforma.

La velocidad vertical inicial se establece en cero:

$$
v_0=0
$$

El desplazamiento inicial del centro de masas también se establece en cero:

$$
z_0=0
$$

La velocidad y el desplazamiento se calculan a partir del inicio detectado del movimiento.

Durante el periodo estable anterior al movimiento, la velocidad y el desplazamiento se mantienen en cero.

---

### 7. Condiciones iniciales para el DJ

En los saltos **drop jump (DJ)**, el participante llega a la plataforma con una velocidad descendente.

Por este motivo, no es apropiado establecer una velocidad igual a cero en el primer contacto.

La velocidad vertical en el contacto se estima a partir de la altura de caída introducida:

$$
v_{contacto}=-\sqrt{2gh}
$$

donde:

* `v_contacto` es la velocidad vertical estimada en el primer contacto;
* `g` es la aceleración de la gravedad;
* `h` es la altura de caída introducida.

El signo negativo representa un movimiento descendente.

El tiempo estimado de caída se calcula mediante:

$$
t_{caida}=\sqrt{\frac{2h}{g}}
$$

La velocidad durante la caída previa al contacto se reconstruye mediante:

$$
v(t)=-gt
$$

La posición durante la caída previa al contacto se reconstruye mediante:

$$
z(t)=-\frac{1}{2}gt^2
$$

Después del primer contacto, la velocidad se calcula integrando la aceleración derivada de la fuerza.

El desplazamiento posterior al contacto se calcula integrando la velocidad.

La altura introducida debería representar, idealmente, el descenso vertical efectivo del centro de masas.

Cuando no se dispone de una medición cinemática externa, la altura del cajón puede utilizarse como aproximación.

Sin embargo, la altura del cajón puede no coincidir exactamente con el descenso real del centro de masas si el participante baja activamente una pierna, flexiona las articulaciones o se impulsa antes de caer.

Por tanto, la trayectoria anterior al contacto debe interpretarse como una reconstrucción estimada.

---

### 8. Detección de los eventos del salto

El análisis identifica los principales eventos temporales del salto.

En CMJ y SJ se detectan:

* inicio del movimiento;
* despegue;
* ápice del vuelo;
* aterrizaje.

En DJ se detectan:

* liberación estimada desde el cajón;
* primer contacto con la plataforma;
* punto más bajo del centro de masas;
* despegue;
* ápice del vuelo;
* aterrizaje.

El punto más bajo del centro de masas durante el contacto se identifica mediante la primera transición de velocidad negativa a velocidad igual o superior a cero:

$$
v(t)<0\quad\longrightarrow\quad v(t)\geq0
$$

Esta transición representa el final del movimiento descendente y el comienzo del movimiento ascendente.

El ápice del vuelo se identifica mediante la primera transición de velocidad positiva a velocidad igual o inferior a cero:

$$
v(t)>0\quad\longrightarrow\quad v(t)\leq0
$$

Si no se detecta esta transición, se utiliza la posición máxima del centro de masas entre el despegue y el aterrizaje.

---

### 9. Altura mediante la velocidad de despegue

La altura del salto se calcula a partir de la velocidad vertical del centro de masas en el despegue:

$$
h_v=\frac{v_{despegue}^2}{2g}
$$

donde:

* `h_v` es la altura calculada mediante la velocidad de despegue;
* `v_despegue` es la velocidad vertical en el despegue;
* `g` es la aceleración de la gravedad.

Este método se aplica a CMJ, SJ y DJ.

La velocidad de despegue procede de la integración de la aceleración calculada a partir de la fuerza vertical.

---

### 10. Altura mediante el desplazamiento del COM

La altura del salto también se calcula a partir del desplazamiento vertical del centro de masas entre el despegue y el ápice:

$$
h_{COM}=z_{apice}-z_{despegue}
$$

donde:

* `h_COM` es la altura calculada mediante el desplazamiento del centro de masas;
* `z_apice` es la posición vertical del centro de masas en el ápice;
* `z_despegue` es la posición vertical del centro de masas en el despegue.

La variable `desplazamiento_COM_desde_despegue_m` se establece en 0 m en el instante del despegue.

En el ápice del vuelo, esta variable alcanza aproximadamente la altura calculada mediante el desplazamiento del centro de masas.

Este método requiere una doble integración y puede ser especialmente sensible a pequeños errores acumulados.

---

### 11. Altura mediante el tiempo de vuelo

Cuando se detectan correctamente el despegue y el aterrizaje, la altura se calcula también mediante el tiempo de vuelo:

$$
h_t=\frac{g·t_{vuelo}^2}{8}
$$

donde:

* `h_t` es la altura calculada mediante el tiempo de vuelo;
* `t_vuelo` es el tiempo transcurrido entre el despegue y el aterrizaje;
* `g` es la aceleración de la gravedad.

Este método supone que la posición vertical del centro de masas es aproximadamente la misma en el despegue y en el aterrizaje.

Si el participante aterriza con una configuración corporal diferente a la utilizada en el despegue, la altura obtenida mediante el tiempo de vuelo puede presentar sesgo.

---

### 12. Comparación entre los métodos de altura

El análisis informa separadamente de:

* altura mediante velocidad de despegue;
* altura mediante desplazamiento del centro de masas;
* altura mediante tiempo de vuelo;
* diferencia entre la altura por desplazamiento y la altura por velocidad;
* diferencia absoluta entre ambos métodos;
* diferencia relativa entre ambos métodos.

La diferencia entre la altura por desplazamiento y la altura por velocidad se calcula mediante:

$$
\Delta h=h_{COM}-h_v
$$

La diferencia absoluta se calcula mediante:

$$
\Delta h_{abs}=\left|h_{COM}-h_v\right|
$$

La diferencia relativa se calcula respecto a la media de ambos métodos:

$$
\Delta h_{rel}=\frac{\left|h_{COM}-h_v\right|}{\left(h_{COM}+h_v\right)/2}\times100
$$

La altura mediante velocidad y la altura mediante desplazamiento no son completamente independientes.

Ambas proceden de la integración de la fuerza vertical.

No obstante, la altura mediante desplazamiento requiere una doble integración, mientras que la altura mediante velocidad requiere una sola integración.

Por este motivo, el método basado en el desplazamiento es más sensible a la deriva acumulada.

---

### 13. Variables específicas del DJ

En el DJ se exportan dos variables diferentes relacionadas con la posición del centro de masas.

#### Desplazamiento relativo al primer contacto

La variable `desplazamiento_COM_rel_contacto_m` se establece en 0 m en el primer contacto con la plataforma.

Esta variable permite analizar:

* descenso del centro de masas después del contacto;
* punto más bajo del centro de masas;
* ascenso durante la fase concéntrica;
* posición del centro de masas en el despegue.

El descenso excéntrico se calcula mediante:

$$
d_{exc}=z_{contacto}-z_{minimo}
$$

El ascenso concéntrico se calcula mediante:

$$
d_{con}=z_{despegue}-z_{minimo}
$$

#### Posición reconstruida desde la liberación

La variable `posicion_COM_reconstruida_desde_liberacion_m` se establece en 0 m en el instante estimado de liberación desde el cajón.

Esta variable incorpora:

* caída balística estimada antes del contacto;
* desplazamiento integrado durante el contacto;
* trayectoria vertical durante el vuelo.

La posición reconstruida en el primer contacto es aproximadamente igual al negativo de la altura de caída introducida.

---

### 14. Fases del DJ

Las fases del DJ se clasifican mediante las siguientes etiquetas:

* `antes_liberacion`
* `caida_previa`
* `fase_excentrica`
* `fase_concentrica`
* `vuelo`
* `post_aterrizaje`

La fase excéntrica comienza en el primer contacto con la plataforma y termina en el punto más bajo del centro de masas.

La duración de la fase excéntrica se calcula mediante:

$$
t_{exc}=t_{minimo}-t_{contacto}
$$

La fase concéntrica comienza en el punto más bajo del centro de masas y termina en el despegue.

La duración de la fase concéntrica se calcula mediante:

$$
t_{con}=t_{despegue}-t_{minimo}
$$

---

### 15. Tiempo de contacto y RSI

En el DJ, el tiempo de contacto se calcula como la diferencia entre el despegue y el primer contacto con la plataforma:

$$
t_{contacto}=t_{despegue}-t_{primer_contacto}
$$

El **reactive strength index (RSI)** principal se calcula utilizando la altura obtenida mediante el tiempo de vuelo:

$$
RSI=\frac{h_t}{t_{contacto}}
$$

También se calculan versiones alternativas utilizando los otros métodos de altura.

El RSI basado en la velocidad de despegue se calcula mediante:

$$
RSI_v=\frac{h_v}{t_{contacto}}
$$

El RSI basado en el desplazamiento del centro de masas se calcula mediante:

$$
RSI_{COM}=\frac{h_{COM}}{t_{contacto}}
$$

Todos los valores del RSI se expresan en m·s⁻¹.

---

### 16. Interpretación de las variables de desplazamiento

En CMJ y SJ, el desplazamiento se expresa respecto al inicio detectado del movimiento.

En DJ se utilizan dos referencias diferentes:

* referencia en el primer contacto;
* referencia en la liberación estimada desde el cajón.

La señal relativa al contacto es la más apropiada para analizar la fase de amortiguación y la fase de propulsión.

La señal reconstruida desde la liberación es útil para representar visualmente la trayectoria completa, pero incorpora una estimación basada en la altura introducida.

Estas dos variables no deben interpretarse como mediciones independientes.

---

### 17. Posibles fuentes de deriva

La integración numérica puede acumular pequeños errores de línea base.

Estos errores afectan especialmente al desplazamiento, porque requiere una doble integración.

La deriva puede estar causada por:

* estimación incorrecta del peso corporal;
* movimiento durante el periodo considerado estable;
* offset en la señal de fuerza;
* inversión incorrecta del signo de la fuerza;
* frecuencia de corte inadecuada;
* detección incorrecta del inicio del movimiento;
* detección incorrecta del primer contacto;
* detección incorrecta del despegue;
* detección incorrecta del ápice;
* detección incorrecta del aterrizaje;
* registros excesivamente largos;
* altura de caída incorrecta en el DJ.

---

### 18. Revisión visual recomendada

Se recomienda revisar visualmente:

* periodo utilizado para estimar el peso corporal;
* estabilidad de la fuerza antes del movimiento;
* fuerza normalizada durante el periodo estable;
* inicio del movimiento;
* primer contacto en el DJ;
* punto más bajo del centro de masas;
* despegue;
* ápice del vuelo;
* aterrizaje;
* velocidad antes del movimiento;
* velocidad en el contacto del DJ;
* velocidad en el despegue;
* posición del centro de masas en el despegue;
* desplazamiento desde el despegue hasta el ápice;
* plausibilidad biomecánica de las curvas;
* concordancia entre los diferentes métodos de cálculo de la altura.

Una discrepancia elevada entre los métodos puede indicar:

* estimación incorrecta del peso corporal;
* deriva acumulada durante la integración;
* detección inadecuada de los eventos;
* diferencia entre las posiciones corporales de despegue y aterrizaje;
* altura de caída incorrecta en el DJ.


#Licencia

68747470733a2f2f692e6372656174697665636f6d6d6f6e732e6f72672f6c2f62792f342e302f38387833312e706e67.png

Este obra está bajo una licencia de [Creative Commons Reconocimiento 4.0 Internacional.](https://creativecommons.org/licenses/by/4.0/)

#Autor

Aarón Miralles Iborra (aaron.mirallesi@umh.es)